## Tools
Models can request to call tools that perform tasks as fetching data from database,searching the web or running code.

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

model=init_chat_model('groq:qwen/qwen3-32b')
model.invoke("Why is sky blue")

AIMessage(content="<think>\nOkay, so the user is asking why the sky is blue. Let me start by recalling what I know about this. I think it has something to do with the way sunlight interacts with the Earth's atmosphere. I remember hearing about Rayleigh scattering being involved. But wait, what exactly is Rayleigh scattering? It's when light is scattered by particles much smaller than the wavelength of the light. The atmosphere is mostly nitrogen and oxygen, which are much smaller than the wavelength of visible light, right? So, sunlight is white, but it's made up of different colors, each with different wavelengths. Blue light has a shorter wavelength compared to red or green. \n\nI think shorter wavelengths are scattered more than longer ones. That means blue light gets scattered in all directions by the molecules in the atmosphere. So, when we look up during the day, we see this scattered blue light. But why isn't the sky violet, since violet has an even shorter wavelength? Maybe our

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the weather of a city """
    return f"It is sunny in {location}"

model_with_tools=model.bind_tools([get_weather])


In [3]:
response=model_with_tools.invoke("Whats the weather in New York")
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in New York. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified New York, I need to call that function with "New York" as the location. I\'ll make sure to format the tool call correctly within the XML tags.\n', 'tool_calls': [{'id': 'j4yfevayk', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 152, 'total_tokens': 247, 'completion_time': 0.130628711, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.005944303, 'prompt_tokens_details': None, 'queue_time': 0.046401286, 'total_time': 0.136573014}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider':

In [5]:
for tool_call in response.tool_calls:
    print(f"The name of the tool:{tool_call['name']}")
    print(f"The args of the tool:{tool_call['args']}")

The name of the tool:get_weather
The args of the tool:{'location': 'New York'}


## Tool execution loops


In [7]:
#We will be passing the result from the tools to Model and ask the question
#step1 - add the result to messages
messages=[{"role":"user","content":"Whats the weather in Hyderabad?"}]
ai_message=model_with_tools.invoke(messages)
messages.append(ai_message)

##Step2 - append the tool calls response from the tools to the messages
#we are enriching the context
for tool_call in ai_message.tool_calls:
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

#step3 - passing the responses to the model
response=model_with_tools.invoke(messages)
response.text

'The weather in Hyderabad is currently sunny. Hope you have a great day! 😊'

In [8]:
response

AIMessage(content='The weather in Hyderabad is currently sunny. Hope you have a great day! 😊', additional_kwargs={'reasoning_content': "Okay, the user asked for the weather in Hyderabad. I called the get_weather function with Hyderabad as the location. The response came back saying it's sunny there. Now I need to relay that information clearly. Let me check if there's any additional details I should mention, but the response seems straightforward. Just state the weather condition and maybe add a friendly note. Make sure it's concise and matches the user's query.\n"}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 191, 'total_tokens': 298, 'completion_time': 0.173978134, 'completion_tokens_details': {'reasoning_tokens': 85}, 'prompt_time': 0.007729706, 'prompt_tokens_details': None, 'queue_time': 0.046562274, 'total_time': 0.18170784}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop

In [9]:
messages

[{'role': 'user', 'content': 'Whats the weather in Hyderabad?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the weather in Hyderabad. Let me check the tools available. There's a function called get_weather that takes a location parameter. I need to call that function with Hyderabad as the location. I'll make sure to format the tool call correctly in JSON inside the XML tags.\n", 'tool_calls': [{'id': 'y9eck3peh', 'function': {'arguments': '{"location":"Hyderabad"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 152, 'total_tokens': 238, 'completion_time': 0.115471282, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.006152901, 'prompt_tokens_details': None, 'queue_time': 0.154647338, 'total_time': 0.121624183}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls